# CE541E08 — Unit 3 · Day 22 — Iterating, Cumulative Operations and Rolling Means

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 22 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | np.cumsum · iterating rows · np.convolve rolling mean · SCS-CN cumulative runoff |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 22"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Why Cumulative and Moving Operations?

So far we have computed statistics across an entire array at once — total, mean, max. Today we focus on operations that **track change over time**:

- **Cumulative sum (`np.cumsum`)** — running total up to each point. Used for: cumulative rainfall, reservoir inflow tracking, total load accumulation.
- **Row iteration** — applying the same analysis to each row of a 2-D array using a for loop (when vectorisation is not straightforward).
- **Rolling mean (`np.convolve`)** — smoothed average over a sliding window. Used for: flood hydrograph smoothing, trend detection, API (Antecedent Precipitation Index).
- **Cumulative SCS-CN runoff** — combining `np.cumsum` and `np.where` to compute incremental runoff from an hourly storm.

---
## Code Block 1 — Cumulative Rainfall with np.cumsum

### What this code does

We compute the running cumulative rainfall total for each day of June 2024. The result tells us not just the daily value but how much rain has fallen in total up to and including each day. We then plot both the daily bars and the cumulative line together.

### Why each step is taken

**`np.cumsum(daily)`:**
`cumsum` stands for cumulative sum. For an array `[a, b, c, d]`, it returns `[a, a+b, a+b+c, a+b+c+d]`. Each output element is the sum of all input elements up to and including that position. This is the standard way to convert a daily rainfall series into a cumulative series — no loop needed.

**Printing only the first 8 days:**
We show just the first 8 days to keep the output readable in the notebook. The full 30-day series is plotted in the chart below.

**`cumulative[-1]`:**
The last element of the cumulative array is always the total sum — equivalent to `daily.sum()`. This is a useful shortcut: after `np.cumsum`, the total is always at index `-1`.

**Dual-axis plot (bars + line):**
The bar chart shows daily rainfall (how much fell each day). The line shows cumulative rainfall (how much has fallen in total). Together they answer two questions: "Was it a rainy day?" and "How much has fallen so far this month?"

### Algorithm

```
1. Create 30-day daily rainfall array (June 2024)

2. cumulative = np.cumsum(daily)
   cumulative[0] = daily[0]
   cumulative[1] = daily[0] + daily[1]
   cumulative[i] = sum of daily[0] through daily[i]

3. Print first 8 days:
   day number, daily value, cumulative total

4. Print month total: cumulative[-1]
   (always equals daily.sum())

5. Plot: bar chart for daily, line for cumulative
   both on the same axes with shared x-axis (day numbers)
```

### Expected output

```
  Day  1:    0.0 mm  cumul:     0.0 mm
  Day  2:    0.0 mm  cumul:     0.0 mm
  Day  3:   12.4 mm  cumul:    12.4 mm
  Day  4:   45.6 mm  cumul:    58.0 mm
  Day  5:    0.0 mm  cumul:    58.0 mm
  Day  6:    8.2 mm  cumul:    66.2 mm
  Day  7:   23.1 mm  cumul:    89.3 mm
  Day  8:    0.0 mm  cumul:    89.3 mm
Month total: 865.5 mm
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# June 2024 daily rainfall (mm) — Cauvery basin
daily = np.array([0, 0, 12.4, 45.6, 0, 8.2, 23.1, 0, 0, 87.3,
                  34.5, 0, 56.2, 0, 18.9, 0, 0, 134.5, 22.3, 45.6,
                  0, 0, 67.8, 12.1, 0, 89.4, 33.2, 0, 45.1, 28.7])

# np.cumsum: running total — each element is the sum of all previous elements + itself
# cumulative[i] = daily[0] + daily[1] + ... + daily[i]
cumulative = np.cumsum(daily)

days = np.arange(1, 31)

# Print first 8 days to show the cumsum building up
for d, r, c in zip(days[:8], daily[:8], cumulative[:8]):
    print(f"  Day {d:>2}: {r:>6.1f} mm  cumul: {c:>7.1f} mm")

# cumulative[-1] is always the grand total (last element of cumsum = total sum)
print(f"Month total: {cumulative[-1]:.1f} mm")

# Dual plot: daily bars + cumulative line
plt.figure(figsize=(10, 4))
plt.bar(days, daily, alpha=0.4, color='steelblue', label='Daily rainfall')
plt.plot(days, cumulative, 'b-o', markersize=3, label='Cumulative')
plt.xlabel('Day of June 2024')
plt.ylabel('Rainfall (mm)')
plt.title('June 2024 — Daily and Cumulative Rainfall')
plt.legend()
plt.tight_layout()
plt.show()

### 🔁 Try this

Find the **first day** when cumulative rainfall exceeded 200 mm.

Use: `np.where(cumulative > 200)[0][0] + 1`

- Which day was it?
- How much had fallen on that day alone?
- What was the running total at that point?

---
## Code Block 2 — Iterating Rows: Year-wise Analysis

### What this code does

We iterate over the rows of a 5-year × 12-month rainfall matrix and compute a custom summary for each year — total, monsoon total, monsoon percentage, and the wettest month name. This combines row iteration with array slicing and the string list lookup.

### Why each step is taken

**`for year, row in zip(years, data)`:**
`zip` pairs each year number with the corresponding row from the matrix. `row` is a 1-D NumPy array of 12 monthly values — so all 1-D array methods work on it directly.

**`row.sum()` — annual total:**
Sums all 12 monthly values for this year. Equivalent to `data[i].sum()` but cleaner inside the loop.

**`row[5:9].sum()` — monsoon total:**
Slices June to September (indices 5 to 8) and sums them. This is the same slice we used in Day 21, now applied to each row in turn.

**`row.argmax()` — index of wettest month:**
Returns the 0-based index of the highest value in the row. We use this as an index into `months_list` to get the month name. For example, if `row.argmax()` returns 5, the wettest month is `months_list[5]` = `'Jun'`.

**Formatted print with f-string alignment:**
`{value:>7}` right-aligns the value in a field 7 characters wide. This makes the table columns line up neatly regardless of value length.

### Algorithm

```
1. Create 5×12 data matrix and year/month label lists

2. Print table header with column widths

3. For each (year, row) pair from zip(years, data):
   a. total   = row.sum()              → annual total
   b. monsoon = row[5:9].sum()         → Jun-Sep total
   c. pct     = monsoon/total*100      → monsoon as % of annual
   d. wet_m   = months_list[row.argmax()] → name of wettest month

4. Print one formatted row per year
```

### Expected output

```
Year   Total   Monsoon      %    Wettest
------------------------------------------
2020     765       452   59.1%       Jun
2021     756       484   64.0%       Jun
2022     783       460   58.8%       Jun
2023     784       465   59.3%       Jun
2024     776       463   59.7%       Jun
```

In [ ]:
import numpy as np

# 5-year × 12-month rainfall matrix (mm)
data = np.array([
    [8, 12,18,52,87,134,118,113,95,71,44,13],   # 2020
    [5,  8,22,48,92,145,123,108,89,68,38,10],   # 2021
    [10,15,16,55,83,128,115,119,98,74,48,16],   # 2022
    [7, 10,20,50,88,138,120,115,92,70,42,12],   # 2023
    [9, 14,19,53,90,140,117,116,96,72,45,14],   # 2024
])

years       = [2020, 2021, 2022, 2023, 2024]
months_list = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

# Print formatted table header
print(f"{'Year':<6} {'Total':>7} {'Monsoon':>9} {'%':>6} {'Wettest':>10}")
print("-" * 42)

# Iterate: zip pairs each year label with its row from the matrix
# 'row' is a 1-D NumPy array — all array methods work on it
for year, row in zip(years, data):
    total   = row.sum()              # sum all 12 months
    monsoon = row[5:9].sum()         # sum Jun(5) Jul(6) Aug(7) Sep(8)
    pct     = monsoon / total * 100  # monsoon as percentage of annual
    wet_m   = months_list[row.argmax()]  # name of month with max value
    print(f"{year:<6} {total:>7} {monsoon:>9} {pct:>5.1f}% {wet_m:>10}")

### 🔁 Try this

Add a final row to show the **5-year mean** for each column.

Use `data.mean(axis=0)` to get the climatology, then compute the same statistics (total, monsoon %, wettest month) for the mean row.

Does the wettest month change across years?

---
## Code Block 3 — Rolling Mean with np.convolve

### What this code does

We compute a 5-day rolling (moving) average of a 30-day streamflow record using `np.convolve`. The rolling mean smooths out day-to-day noise and reveals the underlying trend — rising or falling limb of the hydrograph.

### Why each step is taken

**`np.ones(5)/5` — the averaging kernel:**
A kernel of `[0.2, 0.2, 0.2, 0.2, 0.2]` represents a simple equally-weighted average of 5 consecutive values. When convolved with the data, each output value is the average of the 5 input values centred around (or ending at) that position.

**`np.convolve(flow, kernel, mode='valid')`:**
Convolution slides the kernel across the data array. `mode='valid'` means output is only produced where the kernel fits entirely within the data — no partial windows at the edges. For a 30-element array and a window of 5, the output has `30 - 5 + 1 = 26` values.

**Why the rolling mean starts at Day 5:**
With `mode='valid'` and a 5-day window, the first valid average uses days 1–5. So the rolling mean array aligns with days 5 through 30. `days_roll = np.arange(5, 31)` creates the correct x-axis for plotting.

**Why use a rolling mean at all:**
A single day's streamflow reading may include sensor noise, brief storm spikes, or data errors. The 5-day mean shows the sustained trend — which is what reservoir operators and flood forecasters actually need.

### Algorithm

```
1. Create 30-day streamflow array

2. Define 5-day averaging kernel: np.ones(5)/5
   = [0.2, 0.2, 0.2, 0.2, 0.2]

3. np.convolve(flow, kernel, mode='valid')
   → slides the kernel across flow
   → each output = mean of 5 consecutive input values
   → output length = 30 - 5 + 1 = 26 values

4. days_roll = np.arange(5, 31)
   → x-axis for the rolling mean (starts at Day 5)

5. Plot: raw flow (blue, transparent) + rolling mean (red, solid)
   → rolling mean reveals the trend hidden in noisy daily data

6. Print first 5 rolling mean values for verification
```

### Expected output

```
5-day rolling mean (first 5): [790.  867.8 896.  834.  716. ]
```
*(Plus the plot — verify that the red line is smoother than the blue)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 30-day streamflow record (m³/s) — KRS station, July 2024
flow = np.array([234, 267, 312, 890, 1245, 987, 756, 543, 412, 345,
                 289, 245, 212, 198, 220, 265, 310, 456, 678, 890,
                 1123, 987, 765, 543, 421, 345, 289, 245, 212, 198])

# Averaging kernel: 5 equal weights that sum to 1
# When convolved with data, each output = mean of 5 consecutive values
kernel = np.ones(5) / 5   # [0.2, 0.2, 0.2, 0.2, 0.2]

# mode='valid': only compute where the kernel fits fully inside the data
# Output length = len(flow) - len(kernel) + 1 = 30 - 5 + 1 = 26
rolling = np.convolve(flow, kernel, mode='valid')

days_full = np.arange(1, 31)    # all 30 days for the raw data
days_roll = np.arange(5, 31)    # days 5-30 for the rolling mean (26 values)

plt.figure(figsize=(10, 4))
plt.plot(days_full, flow, 'b-', alpha=0.4, label='Daily flow')
plt.plot(days_roll, rolling, 'r-', linewidth=2, label='5-day rolling mean')
plt.xlabel('Day of July 2024')
plt.ylabel('Flow (m³/s)')
plt.title('Streamflow with 5-Day Rolling Mean — KRS Station')
plt.legend()
plt.tight_layout()
plt.show()

# Print first 5 values of the rolling mean for verification
print("5-day rolling mean (first 5):", np.round(rolling[:5], 1))

### 🔁 Try this

Change the window from 5 days to 3 days and then to 7 days.

- Which window produces a smoother line?
- What happens to the number of output values as the window grows?
- At what window size does the rolling mean start to miss important peaks?

---
## Code Block 4 — Cumulative SCS-CN Runoff from a Storm

### What this code does

We apply the SCS-CN method to a 12-hour design storm to compute cumulative and incremental runoff hour by hour — using `np.cumsum`, `np.where`, and `np.diff` in sequence. No loop is written; the entire computation is vectorised.

### Why each step is taken

**`P_cum = np.cumsum(hourly)`:**
The SCS-CN formula works on cumulative rainfall, not hourly rainfall. `np.cumsum` converts the hourly values into a running total — the cumulative storm depth at the end of each hour.

**`S = 25400/CN - 254` and `Ia = 0.2 * S`:**
S is the potential maximum retention (mm) — how much water the soil can absorb. For CN=75, S = 84.7 mm. Ia is the initial abstraction (mm) — the rainfall that must fall before any runoff begins. For CN=75, Ia = 16.9 mm. No runoff occurs until cumulative rainfall exceeds Ia.

**`Q_cum = np.where(P_cum > Ia, (P_cum-Ia)**2/(P_cum-Ia+S), 0)`:**
The SCS-CN runoff formula applied vectorially to the entire cumulative rainfall array at once. `np.where(condition, value_if_true, value_if_false)` replaces an if-else statement for each hour.

**`Q_hr = np.diff(Q_cum, prepend=0)`:**
`np.diff` computes the difference between consecutive elements — giving the incremental runoff in each hour. `prepend=0` adds a zero before the first element so the output length matches the input length.

### Algorithm

```
1. Define 12-hour storm rainfall (mm/hr)

2. Set CN=75, compute:
   S  = 25400/CN - 254  (potential retention, mm)
   Ia = 0.2 * S         (initial abstraction, mm)

3. P_cum = np.cumsum(hourly)
   → cumulative storm depth at end of each hour

4. Q_cum = np.where(P_cum > Ia,
               (P_cum - Ia)**2 / (P_cum - Ia + S),
               0)
   → cumulative runoff at end of each hour
   → zero until P_cum exceeds Ia

5. Q_hr = np.diff(Q_cum, prepend=0)
   → incremental runoff each hour
   → np.diff computes Q_cum[i] - Q_cum[i-1]

6. Print table: hour, P_hourly, P_cum, Q_cum, Q_hr
7. Print storm totals: total rain vs total runoff
```

### Expected output

```
CN=75, S=84.7 mm, Ia=16.9 mm
  Hr   P_hrly    P_cum    Q_cum     Q_hr
----------------------------------------
   1      2.1      2.1     0.00     0.00
   2      4.5      6.6     0.00     0.00
   3      8.9     15.5     0.00     0.00
   4     15.6     31.1     3.99     3.99
   5     22.3     53.4    14.87    10.88
   6     31.4     84.8    33.14    18.27
   7     28.7    113.5    52.91    19.77
   8     18.9    132.4    67.24    14.33
   9     12.3    144.7    77.42    10.18
  10      7.8    152.5    84.11     6.69
  11      4.2    156.7    87.41     3.30
  12      2.1    158.8    89.12     1.71
Total: 158.8 mm rain, 89.12 mm runoff
```

In [ ]:
import numpy as np

# 12-hour storm — hourly rainfall (mm/hr)
hourly = np.array([2.1, 4.5, 8.9, 15.6, 22.3, 31.4,
                   28.7, 18.9, 12.3, 7.8, 4.2, 2.1])

# SCS-CN parameters
CN = 75
S  = 25400/CN - 254   # potential maximum retention (mm)
Ia = 0.2 * S          # initial abstraction (mm) — rainfall before runoff starts
print(f"CN={CN}, S={S:.1f} mm, Ia={Ia:.1f} mm")

# Step 1: cumulative storm depth at end of each hour
P_cum = np.cumsum(hourly)

# Step 2: cumulative runoff using SCS-CN formula
# np.where applies the formula where P_cum > Ia, and returns 0 otherwise
# This replaces an if-else statement for each of the 12 hours
Q_cum = np.where(P_cum > Ia,
                 (P_cum - Ia)**2 / (P_cum - Ia + S),
                 0)

# Step 3: incremental (hourly) runoff = difference between consecutive cumulative values
# np.diff computes Q_cum[i] - Q_cum[i-1]; prepend=0 keeps the output length = 12
Q_hr = np.diff(Q_cum, prepend=0)

# Print hour-by-hour table
print(f"{'Hr':>4} {'P_hrly':>8} {'P_cum':>8} {'Q_cum':>8} {'Q_hr':>8}")
print("-" * 40)
for i in range(len(hourly)):
    print(f"{i+1:>4} {hourly[i]:>8.1f} {P_cum[i]:>8.1f} {Q_cum[i]:>8.2f} {Q_hr[i]:>8.2f}")

print(f"Total: {hourly.sum():.1f} mm rain, {Q_cum[-1]:.2f} mm runoff")

### 🔁 Try this

Change `CN = 75` to `CN = 90` (more impervious catchment — urban area).

- How does the initial abstraction Ia change?
- Does runoff start earlier or later?
- How much does total runoff increase?

---
## Session Summary — Cumulative and Rolling Operations

| Function | What it does | Example |
|---|---|---|
| `np.cumsum(arr)` | Running total — each element = sum of all previous + itself | `np.cumsum([2,3,5])` → `[2,5,10]` |
| `cumulative[-1]` | Last element of cumsum = grand total | Always equals `arr.sum()` |
| `np.convolve(arr, kernel, mode)` | Slides kernel across array — used for rolling means | `np.convolve(flow, np.ones(5)/5, 'valid')` |
| `mode='valid'` | Output only where kernel fits fully inside data | Length = `len(arr) - len(kernel) + 1` |
| `np.diff(arr, prepend=0)` | Consecutive differences — reverses cumsum | `np.diff([0,3,5,8])` → `[3,2,3]` |
| `np.where(cond, a, b)` | Element-wise if-else on arrays | `np.where(P>Ia, formula, 0)` |
| `zip(labels, data)` | Pair label list with matrix rows for iteration | `for year, row in zip(years, data)` |
| `row.argmax()` | Index of maximum value in a row | Used to look up month name |

---
## Day 22 Assignment

Use the June 2024 daily rainfall from Code Block 1.

1. Compute cumulative rainfall and find the **first day cumulative exceeds 200 mm**
2. Compute a **3-day rolling mean** and a **7-day rolling mean** — print the first 5 values of each
3. For each day from Day 6 onwards, compute the **5-day Antecedent Precipitation Index (API5)** = sum of the previous 5 days' rainfall. Print a table showing day number and API5.

### ▶ Assignment cell

In [ ]:
import numpy as np

daily = np.array([0, 0, 12.4, 45.6, 0, 8.2, 23.1, 0, 0, 87.3,
                  34.5, 0, 56.2, 0, 18.9, 0, 0, 134.5, 22.3, 45.6,
                  0, 0, 67.8, 12.1, 0, 89.4, 33.2, 0, 45.1, 28.7])

# 1. Cumulative and day > 200 mm
cumul   = np.cumsum(daily)
day_200 = ???   # first day cumul > 200 (1-based)
print(f"Cumul > 200mm first on Day {day_200}")

# 2. Rolling means
roll3 = np.convolve(daily, np.ones(3)/3, mode='valid')
roll7 = np.convolve(daily, np.ones(7)/7, mode='valid')
print(f"3-day rolling mean (first 5): {np.round(roll3[:5], 1)}")
print(f"7-day rolling mean (first 5): {np.round(roll7[:5], 1)}")

# 3. API5: sum of previous 5 days for each day from Day 6 onwards
print(f"{'Day':>4} {'API5(mm)':>10}")
for i in range(5, len(daily)):
    api5 = daily[i-5:i].sum()   # sum of days i-5 to i-1 (previous 5 days)
    print(f"{i+1:>4} {api5:>10.1f}")

---
- [ ] Run all cells from top to bottom — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholder)
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day22.ipynb`
- [ ] Commit message: `Day 22 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*